In [ ]:
from roboflow import Roboflow
import os
import shutil
import random

# 1. Download from Roboflow (Export as YOLOv8 Instance Segmentation)
rf = Roboflow(api_key="9X8ClVkM9agUJAH3hcAN")
project = rf.workspace("hard-drive").project("hdd_global_scout")
version = project.version(3)
dataset = version.download("yolov8")

DATA_PATH = dataset.location
OUTPUT_DIR = "dataset_rfdetr_seg"

# 2. Create directory structure
for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(OUTPUT_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, split, 'labels'), exist_ok=True)

images_dir = os.path.join(DATA_PATH, "train", "images")
labels_dir = os.path.join(DATA_PATH, "train", "labels")

# 3. Shuffle and split (80% Train, 10% Val, 10% Test)
images = [f for f in os.listdir(images_dir) if f.endswith('.jpg')]
random.seed(42)
random.shuffle(images)

train_idx, val_idx = int(len(images) * 0.8), int(len(images) * 0.9)
splits = {'train': images[:train_idx], 'valid': images[train_idx:val_idx], 'test': images[val_idx:]}

# 4. Safely move Image/Label pairs
for split_name, img_list in splits.items():
    for img_name in img_list:
        shutil.copy(os.path.join(images_dir, img_name), os.path.join(OUTPUT_DIR, split_name, 'images', img_name))
        label_name = img_name.replace('.jpg', '.txt')
        if os.path.exists(os.path.join(labels_dir, label_name)):
            shutil.copy(os.path.join(labels_dir, label_name), os.path.join(OUTPUT_DIR, split_name, 'labels', label_name))

print(f"✅ Dataset split successfully into {OUTPUT_DIR}/")

In [ ]:
import yaml
import os

# Define the folder where you moved the split images
OUTPUT_DIR = "dataset_rfdetr_seg"

# Since you ran the download cell earlier, DATA_PATH should still be in memory. 
# If not, replace it with the absolute path to your original roboflow download folder.
original_yaml_path = os.path.join(DATA_PATH, 'data.yaml')

# Load the original class names from Roboflow's YAML
with open(original_yaml_path, 'r') as f:
    data_yaml = yaml.safe_load(f)

# Update paths to match the internal structure of OUTPUT_DIR
# RF-DETR prefers simple relative paths from the location of the data.yaml
data_yaml['train'] = 'train/images'
data_yaml['val'] = 'valid/images' 
data_yaml['test'] = 'test/images'

# Save the new data.yaml directly INSIDE the output directory
new_yaml_path = os.path.join(OUTPUT_DIR, 'data.yaml')
with open(new_yaml_path, 'w') as f:
    yaml.dump(data_yaml, f)

print(f"✅ Fixed! data.yaml successfully created at: {new_yaml_path}")
print(f"Detected {len(data_yaml['names'])} classes: {data_yaml['names']}")

In [ ]:
import supervision as sv
import cv2
import matplotlib.pyplot as plt

# Load dataset into Supervision to verify segmentation masks
dataset = sv.DetectionDataset.from_yolo(
    images_directory_path=os.path.join(OUTPUT_DIR, 'train', 'images'),
    annotations_directory_path=os.path.join(OUTPUT_DIR, 'train', 'labels'),
    data_yaml_path=os.path.join(DATA_PATH, 'data.yaml')
)

image_name, image, annotations = dataset[0]

# Setup Annotators
mask_annotator = sv.MaskAnnotator(opacity=0.6)
box_annotator = sv.BoxAnnotator(thickness=2)

# Apply Annotations
annotated_image = mask_annotator.annotate(scene=image.copy(), detections=annotations)
annotated_image = box_annotator.annotate(scene=annotated_image, detections=annotations)

plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title("Ground Truth Verification")
plt.show()

In [ ]:
from rfdetr import RFDETRSegMedium

# Initialize the segmentation variant of RF-DETR
model = RFDETRSegMedium()

# Train the model
model.train(
    dataset_dir=OUTPUT_DIR,
    epochs=100,
    batch_size=2,              # Safe for 8GB VRAM at high resolution
    grad_accum_steps=8,        # Accumulate gradients to maintain stable learning
    resolution=1024            # DINOv2 backbone works best with square resolutions
)

In [ ]:
import pandas as pd
import os

# RF-DETR logs its metrics in the output directory
res_path = "runs/train/exp/" # Check your terminal for the exact 'exp' folder name
log_csv = os.path.join(res_path, 'metrics.csv')

if os.path.exists(log_csv):
    df = pd.read_csv(log_csv)
    # Get the best validation mAP for segmentation masks
    best_mask_map = df['val/seg_mAP_50_95'].max()
    print(f"🏆 Best Mask mAP (50-95): {best_mask_map:.4f}")
else:
    print("Metrics file not found. Ensure training completed.")

In [ ]:
if os.path.exists(log_csv):
    plt.figure(figsize=(18, 5))

    # --- 1. Segmentation Loss ---
    plt.subplot(1, 3, 1)
    plt.plot(df['epoch'], df['train/loss_mask'], label='Train Mask Loss')
    plt.plot(df['epoch'], df['val/loss_mask'], label='Val Mask Loss', linestyle='--')
    plt.title('RF-DETR Mask Loss')
    plt.xlabel('Epoch')
    plt.legend()

    # --- 2. Box Loss ---
    plt.subplot(1, 3, 2)
    plt.plot(df['epoch'], df['train/loss_bbox'], label='Train Box Loss')
    plt.plot(df['epoch'], df['val/loss_bbox'], label='Val Box Loss', linestyle='--')
    plt.title('RF-DETR Bounding Box Loss')
    plt.xlabel('Epoch')
    plt.legend()

    # --- 3. Segmentation mAP ---
    plt.subplot(1, 3, 3)
    plt.plot(df['epoch'], df['val/seg_mAP_50_95'], label='Mask mAP 50-95', color='green', linewidth=2)
    plt.title('Segmentation mAP (Accuracy)')
    plt.xlabel('Epoch')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
from PIL import Image

# Load the trained checkpoint
model = RFDETRSegMedium(pretrain_weights=os.path.join(res_path, "weights/best.pt"), resolution=1024)
model.optimize_for_inference() # Required for maximum FPS

# Grab a test image
test_images_dir = os.path.join(OUTPUT_DIR, 'test', 'images')
test_img_path = os.path.join(test_images_dir, os.listdir(test_images_dir)[0])
pil_image = Image.open(test_img_path)

# Run Inference
detections = model.predict(pil_image, threshold=0.4)

# Setup Annotators
color_palette = sv.ColorPalette.DEFAULT
mask_annotator = sv.MaskAnnotator(color=color_palette)
box_annotator = sv.BoxAnnotator(color=color_palette, thickness=2)

# Convert PIL image to CV2 format for drawing
cv_image = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)

# Apply Annotations (RF-DETR predict output plugs directly into SV)
annotated_image = mask_annotator.annotate(scene=cv_image.copy(), detections=detections)
annotated_image = box_annotator.annotate(scene=annotated_image, detections=detections)

# Display High-Res Result
plt.figure(figsize=(12, 12))
plt.imshow(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title("RF-DETR Segmentation Inference (1024px)")
plt.show()